## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. The first cell installs what's needed,
downloads the camp toolbox, and downloads the data — just press ▶ and wait for
the green **✅ Setup complete**, then run the rest of the notebook top to bottom.

It also offers to connect your Google Drive so your figures are *saved* for your
poster (recommended). If you skip that, the notebook still works — your figures
just won't persist after you close Colab.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os

print("1/3  installing libraries ...")
get_ipython().system('pip install -q "mne==1.10.1" gdown')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

print("3/3  downloading the data (~470 MB, first time only) ...")
import gdown
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/synapse_preprocessed.pkl"):
    gdown.download(id="1Z-NENlKMjL-kL-N46lQ8QA1AbGM7bJHY",
                   output="data/synapse_preprocessed.pkl", quiet=False)
os.environ["CAMP_DATA_PATH"] = "data/synapse_preprocessed.pkl"

# Save figures to your own Drive so they persist for your poster (recommended).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
    where = "Drive > DecodingBrain_outputs"
except Exception:
    os.environ["CAMP_OUTPUT_DIR"] = "outputs"
    where = "a temporary 'outputs' folder (download anything you want to keep!)"
print(f"\n\u2705 Setup complete. Figures will be saved to {where}.")


# Week 2 · Day 10 — Clinical Correlations  🏁 **Checkpoint 2**

Group differences tell us *that* sound-sensitive brains differ. This notebook
asks something deeper:

> **Do people with *worse symptoms* show *bigger brain differences*?**

If a brain feature tracks symptom severity, it's a much stronger candidate for a
real **biomarker**. We'll correlate EEG features with clinical questionnaire
scores (this is **Research Goal 3** from the kickoff deck).

### By the end of this notebook you will be able to
1. Join EEG features with clinical scores
2. Compute a Spearman rank correlation and read it
3. Make a scatter plot with a trend line
4. Correct correlations for multiple comparisons, and stay appropriately humble

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
import camp_utils as cu

data = cu.load_camp_data(verbose=False)
features = pd.read_csv(cu.save_path("features_table.csv"))

## 1. The questionnaires
Only the **EXP** group filled these out (controls have no symptoms to rate).
Higher score = worse symptoms.

| Score | Measures |
|---|---|
| `HQ_Total` | hyperacusis severity (sound sensitivity overall) |
| `GAD_Total` | anxiety (GAD-7) |
| `THI_Total` | tinnitus handicap |
| `Miso_Section1` | misophonia |

In [ ]:
for measure in ["HQ_Total", "GAD_Total", "THI_Total", "Miso_Section1"]:
    print(f"  {measure:14s} — {cu.CLINICAL_MEASURES[measure]}")

## 2. Attach a clinical score to the feature table
We add a column to the EXP rows giving each subject's score. (CTRL rows get NaN
and drop out of the correlation automatically.)

In [ ]:
def add_clinical_column(features, data, measure):
    """Return a copy of `features` with a new column for the clinical score."""
    scores = []
    for subj in features["subject"]:
        scores.append(cu.get_clinical_score(data, subj, measure))
    out = features.copy()
    out[measure] = scores
    return out

df = add_clinical_column(features, data, "HQ_Total")
print("Subjects with an HQ score:", df["HQ_Total"].notna().sum())
df[["subject", "group", "let_gamma", "HQ_Total"]].head(8)

## 3. Spearman correlation
We use **Spearman** (rank-based) correlation, like the study does, because it's
robust to outliers and doesn't assume a straight-line relationship — it just asks
*"as one goes up, does the other tend to go up (or down)?"*

- **r near +1** → strong positive (more symptoms, more of this brain feature)
- **r near −1** → strong negative
- **r near 0** → no relationship

In [ ]:
sub = df.dropna(subset=["let_gamma", "HQ_Total"])
r, p = stats.spearmanr(sub["let_gamma"], sub["HQ_Total"])
print(f"LET gamma  vs  HQ_Total:   r = {r:+.2f},  p = {p:.3f}   (n = {len(sub)})")

## 4. Scatter plot with a trend line
Always *look* at a correlation — a single number can hide a lot.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(sub["HQ_Total"], sub["let_gamma"], color=cu.EXP_COLOR, s=60, alpha=0.8)
# add a simple least-squares trend line
slope, intercept = np.polyfit(sub["HQ_Total"], sub["let_gamma"], 1)
xline = np.array([sub["HQ_Total"].min(), sub["HQ_Total"].max()])
ax.plot(xline, slope * xline + intercept, color="black", linestyle="--")
ax.set_xlabel("Hyperacusis Questionnaire score (worse →)")
ax.set_ylabel("LET gamma power change (dB)")
ax.set_title(f"Brain–symptom link   (r = {r:+.2f}, p = {p:.3f})")
plt.tight_layout()
plt.show()

### ✏️ Your turn #1 — write a correlation function
Fill in this helper so it returns `(r, p, n)` for any feature vs any measure.

In [ ]:
def correlate(df_with_score, feature, measure):
    """Spearman correlation of a feature with a clinical score. Returns (r, p, n)."""
    sub = df_with_score.dropna(subset=[feature, measure])
    # TODO: run stats.spearmanr(sub[feature], sub[measure]) -> r, p
    r, p = None, None
    return r, p, len(sub)

r2, p2, n2 = correlate(df, "let_beta", "HQ_Total")
cu.check(r2 is not None,
         f"let_beta vs HQ_Total: r = {r2:+.2f}, p = {p2:.3f}, n = {n2}" if r2 is not None else "",
         "Call stats.spearmanr(sub[feature], sub[measure]).")

## 5. Search across many features and measures
Let's correlate several EEG features with several clinical scores and find the
strongest relationships.

In [ ]:
eeg_features = ["let_gamma", "let_alpha", "let_beta", "ast_theta", "let_n1_amp"]
measures = ["HQ_Total", "GAD_Total", "THI_Total", "Miso_Section1"]

rows = []
for measure in measures:
    dfm = add_clinical_column(features, data, measure)
    for feat in eeg_features:
        r, p, n = correlate(dfm, feat, measure)
        rows.append({"feature": feat, "measure": measure, "r": r, "p": p, "n": n})

corr = pd.DataFrame(rows)
# FDR-correct: we just ran many correlations!
corr = corr.dropna(subset=["p"])
_, q, _, _ = multipletests(corr["p"].values, alpha=0.05, method="fdr_bh")
corr["q_value"] = q
print(corr.sort_values("p").round(3).to_string(index=False))

### ✏️ Your turn #2 — the strongest link
Find the row with the largest **absolute** correlation `r`.

In [ ]:
# TODO: strongest = the row of `corr` with the biggest |r|
#   hint: corr.loc[corr["r"].abs().idxmax()]
strongest = None

cu.check(strongest is not None,
         f"Strongest link: {strongest['feature']} vs {strongest['measure']} "
         f"(r = {strongest['r']:+.2f})" if strongest is not None else "",
         "Use corr['r'].abs().idxmax() to get the row label, then corr.loc[...].")

## 6. Stay humble — a vital lesson
With only ~18 EXP subjects, correlations are **fragile**: one unusual person can
swing `r` a lot. The SYNAPSE study labels *every* clinical correlation as
**exploratory**, no matter how pretty, because:
- small samples give unstable estimates,
- we tested many combinations (multiple comparisons),
- correlation never proves causation.

Good scientists report these as "interesting leads for future work," not proof.
Write that framing on your poster.

In [ ]:
corr.sort_values("p").to_csv(cu.save_path("clinical_correlations.csv"), index=False)
print("Saved to outputs/clinical_correlations.csv")

## 🏁 Checkpoint 2 — show your instructor
- [ ] A scatter plot with a trend line for one brain–symptom relationship
- [ ] Your `correlate()` function works
- [ ] You can name your strongest correlation **and** explain why it's only
      "exploratory"

➡️ **Next week:** you choose a **Tier** and go deep — spatial brain maps,
subgroups, time dynamics, or machine-learning classification.